# Infrared Solar Physics — Implementation / 적외선 태양물리 구현

**Paper**: M. J. Penn, *Living Rev. Solar Phys.* 11, 2 (2014).  
**DOI**: 10.12942/lrsp-2014-2

---

## Overview / 개요

**English.** This notebook reproduces the key quantitative arguments of Penn's 2014 *Living Reviews* article on infrared solar physics. Each of the six sections below is a standalone numerical or graphical experiment illustrating one of the advantages the paper highlights:

1. **Zeeman splitting** — compare Fe I 1565 nm vs. optical Fe I 5250 Å across a range of field strengths.
2. **Planck function & Rayleigh-Jeans limit** — show how the solar spectrum falls off in the IR and when RJ is valid.
3. **CO 4.7 μm synthetic line profile** — model absorption lines of the fundamental CO band.
4. **Atmospheric transmission windows** — approximate the near/mid-IR telluric transmission.
5. **Coronal [Fe XIII] 1074.7 nm visibility** — demonstrate the off-limb scattered-light advantage of IR coronal observations.
6. **Magnetic-sensitivity figure of merit** — reproduce the $g_{\rm eff}\lambda$ ranking of Table 2.

**한국어.** 이 노트북은 Penn(2014) *Living Reviews* 논문의 정량적 주장을 재현한다. 아래 여섯 절은 각각 독립적인 수치·도식 실험이며 논문이 강조하는 IR의 이점들을 시각적으로 확인한다:

1. **제만 분리** — Fe I 1565 nm 대 가시광 Fe I 5250 Å을 자기장 범위에서 비교.
2. **플랑크 함수와 Rayleigh-Jeans 극한** — IR에서 태양 스펙트럼의 감소와 RJ 유효 구간.
3. **CO 4.7 μm 합성 선 프로파일** — 기본 CO 밴드의 흡수선 모델.
4. **대기 투과 창** — 근/중 IR 대기 투과율 근사.
5. **코로나 [Fe XIII] 1074.7 nm 가시성** — 림 바깥 산란광 이점 시연.
6. **자기 민감도 지표** — Table 2의 $g_{\rm eff}\lambda$ 순위 재현.

## Setup / 설정

**English.** Standard scientific Python stack. All code comments are Google-style English per project convention.  
**한국어.** 표준 Python 과학계산 스택을 사용한다. 코드 주석은 프로젝트 규약에 따라 Google 스타일 영어로 작성한다.

In [ ]:
# Core imports for IR solar physics calculations.
import numpy as np
import matplotlib.pyplot as plt

# Physical constants (SI).
H_PLANCK = 6.62607015e-34  # J s
C_LIGHT = 2.99792458e8     # m/s
K_BOLTZ = 1.380649e-23     # J/K
E_CHARGE = 1.602176634e-19 # C
M_E = 9.1093837015e-31     # kg

# Matplotlib cosmetic defaults.
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 10,
})

print('Setup complete. NumPy', np.__version__)

## 1. Zeeman Splitting: IR vs. Optical / 제만 분리: IR 대 가시광

**English.** The Zeeman displacement of $\sigma$ components from line center in a longitudinal magnetic field $B$ is

$$\Delta\lambda_B = \frac{e}{4\pi m_e c^2}\, g_{\rm eff}\, \lambda^2\, B \approx 4.67\times 10^{-5}\, g_{\rm eff}\, \lambda_{\rm nm}^2\, B_{\rm G}\ [\mathrm{m\mathring{A}}]$$

The $\lambda^2$ scaling is the central IR advantage (Penn 2014 §2.3.1). We compare Fe I 1565 nm ($g=3$) with optical Fe I 5250 Å ($g=3$) for $B = 100$ G to 3000 G.

**한국어.** 시선방향 자기장 $B$에서 $\sigma$ 성분의 제만 이동은 위 식으로 주어진다. $\lambda^2$ 의존성이 IR의 핵심 이점이다(Penn 2014 §2.3.1). Fe I 1565 nm($g=3$)와 가시광 Fe I 5250 Å($g=3$)을 $B = 100$–$3000$ G 범위에서 비교한다.

In [ ]:
def zeeman_split_mA(lambda_nm, g_eff, B_gauss):
    """Compute the Zeeman sigma-component displacement in milli-angstroms.

    Args:
        lambda_nm: Rest wavelength in nanometers.
        g_eff: Effective Lande factor of the transition.
        B_gauss: Longitudinal magnetic field in gauss.

    Returns:
        Displacement of the sigma component from line center in milli-angstroms.
    """
    return 4.67e-5 * g_eff * (lambda_nm ** 2) * B_gauss


def doppler_width_mA(lambda_nm, T_kelvin=6000.0, xi_kms=1.0, atomic_mass_amu=56.0):
    """Approximate thermal+microturbulent Doppler width in milli-angstroms.

    Args:
        lambda_nm: Rest wavelength in nanometers.
        T_kelvin: Plasma temperature in K (default 6000 for photosphere).
        xi_kms: Microturbulent velocity in km/s.
        atomic_mass_amu: Atomic mass in amu (Fe = 56).

    Returns:
        Doppler width (1/e half-width) in milli-angstroms.
    """
    m = atomic_mass_amu * 1.66053906660e-27
    v_th = np.sqrt(2.0 * K_BOLTZ * T_kelvin / m) / 1e3  # km/s
    v_total = np.sqrt(v_th**2 + xi_kms**2)
    lambda_angstrom = lambda_nm * 10.0
    return lambda_angstrom * (v_total * 1e3) / C_LIGHT * 1e3


# Sweep magnetic field range
B_range = np.linspace(100, 3000, 200)

opt_lambda, opt_g = 525.022, 3.0
ir_lambda, ir_g = 1564.85, 3.0

split_opt = zeeman_split_mA(opt_lambda, opt_g, B_range)
split_ir = zeeman_split_mA(ir_lambda, ir_g, B_range)

doppler_opt = doppler_width_mA(opt_lambda)
doppler_ir = doppler_width_mA(ir_lambda)

print(f'Doppler width at 525.022 nm, 6000 K, Fe: {doppler_opt:.1f} mA')
print(f'Doppler width at 1564.85 nm, 6000 K, Fe: {doppler_ir:.1f} mA')
print('At B = 1000 G:')
print(f'  Optical split = {zeeman_split_mA(opt_lambda, opt_g, 1000):.1f} mA '
      f'(ratio to Doppler: {zeeman_split_mA(opt_lambda, opt_g, 1000)/doppler_opt:.2f})')
print(f'  IR split      = {zeeman_split_mA(ir_lambda, ir_g, 1000):.1f} mA '
      f'(ratio to Doppler: {zeeman_split_mA(ir_lambda, ir_g, 1000)/doppler_ir:.2f})')

In [ ]:
# Plot comparison of Zeeman splitting vs B for optical and IR lines.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

ax = axes[0]
ax.plot(B_range, split_opt, lw=2, color='C0', label='Fe I 5250.22 A (optical)')
ax.plot(B_range, split_ir, lw=2, color='C3', label='Fe I 15648.5 A (IR)')
ax.axhline(doppler_opt, ls='--', color='C0', alpha=0.5,
           label=f'Doppler width (opt) = {doppler_opt:.0f} mA')
ax.axhline(doppler_ir, ls='--', color='C3', alpha=0.5,
           label=f'Doppler width (IR)  = {doppler_ir:.0f} mA')
ax.set_xlabel('Magnetic field B [G]')
ax.set_ylabel('Zeeman sigma displacement [mA]')
ax.set_title('IR Zeeman splitting: 1565 nm vs. 5250 A')
ax.legend(loc='upper left', fontsize=8)
ax.set_yscale('log')

ax = axes[1]
ax.plot(B_range, split_opt / doppler_opt, lw=2, color='C0', label='Optical')
ax.plot(B_range, split_ir / doppler_ir, lw=2, color='C3', label='IR')
ax.axhline(1.0, ls=':', color='k', alpha=0.5, label='Onset of full splitting')
ax.set_xlabel('Magnetic field B [G]')
ax.set_ylabel('Zeeman split / Doppler width')
ax.set_title('Magnetic resolution vs. B')
ax.legend(loc='upper left', fontsize=9)

plt.tight_layout()
plt.show()

**English.** The IR line saturates the Doppler width near $B \sim 200$ G and is *fully split* by $\sim 1$ kG (ratio > 4). The optical line remains in the weak-field regime throughout: even at 3 kG its split is only about twice the Doppler width, so visible magnetograms measure $B \cdot f$ (filling-factor × field) rather than $B$ directly. This is precisely the argument Stenflo et al. (1987) used to justify the IR — see Penn (2014) Figure 4.

**한국어.** IR 선은 약 200 G에서 이미 도플러 폭을 넘고 1 kG 근처에서 *완전히 분리*된다(비율 > 4). 반면 가시광 선은 3 kG에서도 약자기장 영역에 머무르므로 가시광 자기도는 $B \cdot f$(filling factor × 자기장)만 측정한다. 이것이 Stenflo et al.(1987)의 핵심 논증이며 Penn(2014) Figure 4와 일치한다.

## 2. Planck Function at IR Wavelengths / IR 파장에서의 플랑크 함수

**English.** The full blackbody spectral radiance is

$$B_\lambda(T) = \frac{2hc^2}{\lambda^5}\,\frac{1}{\exp(hc/\lambda k_B T) - 1}.$$

In the long-wavelength limit ($hc/\lambda k_B T \ll 1$) this reduces to the Rayleigh-Jeans law $B_\lambda \to 2ck T/\lambda^4$. We compare full Planck and RJ for solar-relevant wavelengths $\lambda = 1.5, 5, 12\,\mu$m vs. temperature (Penn 2014 §2.4.1).

**한국어.** 완전한 흑체 복사는 위 플랑크 식으로 주어지며, 장파장 극한($hc/\lambda k_B T \ll 1$)에서 Rayleigh-Jeans 법칙 $B_\lambda \to 2ck T/\lambda^4$로 수렴한다. 태양 관련 파장 $\lambda = 1.5, 5, 12\,\mu$m에서 완전 플랑크 함수와 RJ 근사를 온도에 대해 비교한다(Penn 2014 §2.4.1).

In [ ]:
def planck_lambda(wavelength_m, temperature_k):
    """Evaluate the full Planck spectral radiance B_lambda(T).

    Args:
        wavelength_m: Wavelength in meters (scalar or array).
        temperature_k: Temperature in kelvin (scalar or array).

    Returns:
        B_lambda in W m^-2 sr^-1 m^-1.
    """
    x = H_PLANCK * C_LIGHT / (wavelength_m * K_BOLTZ * temperature_k)
    return (2.0 * H_PLANCK * C_LIGHT**2 / wavelength_m**5) / np.expm1(x)


def rayleigh_jeans_lambda(wavelength_m, temperature_k):
    """Rayleigh-Jeans long-wavelength approximation.

    Args:
        wavelength_m: Wavelength in meters.
        temperature_k: Temperature in kelvin.

    Returns:
        Approximate B_lambda in W m^-2 sr^-1 m^-1.
    """
    return 2.0 * C_LIGHT * K_BOLTZ * temperature_k / wavelength_m**4


T_range = np.linspace(3000, 8000, 400)
wavelengths = {'1.5 um (near-IR)': 1.5e-6, '5 um (mid-IR)': 5e-6, '12 um (mid-IR)': 12e-6}

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

ax = axes[0]
for lbl, w in wavelengths.items():
    B = planck_lambda(w, T_range)
    ax.plot(T_range, B, lw=2, label=lbl)
ax.set_xlabel('Temperature [K]')
ax.set_ylabel(r'B$_\lambda$ [W m$^{-2}$ sr$^{-1}$ m$^{-1}$]')
ax.set_title('Planck function at IR wavelengths')
ax.set_yscale('log')
ax.legend(fontsize=9)
ax.axvline(5800, ls=':', color='orange', label='Solar Teff')

ax = axes[1]
lambdas = np.logspace(-7, -4, 500)
T_solar = 5800.0
B_full = planck_lambda(lambdas, T_solar)
B_rj = rayleigh_jeans_lambda(lambdas, T_solar)
ax.plot(lambdas * 1e6, B_full, lw=2, label='Full Planck')
ax.plot(lambdas * 1e6, B_rj, lw=1.5, ls='--', label='Rayleigh-Jeans')
ax.axvspan(1.0, 12.5, color='C3', alpha=0.12, label='IR window (1.0-12.5 um)')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Wavelength [um]')
ax.set_ylabel(r'B$_\lambda$ [W m$^{-2}$ sr$^{-1}$ m$^{-1}$]')
ax.set_title(f'Planck vs. Rayleigh-Jeans at T = {T_solar:.0f} K')
ax.legend(fontsize=9, loc='lower left')
ax.set_ylim(1e6, 1e14)

plt.tight_layout()
plt.show()

for lbl, w in wavelengths.items():
    r = planck_lambda(w, T_solar) / rayleigh_jeans_lambda(w, T_solar)
    print(f'Planck/RJ ratio at {lbl}, T=5800 K: {r:.3f}')

**English.** The RJ approximation becomes accurate only beyond $\sim 5\,\mu$m at solar temperatures; at 1.5 μm the full Planck is $\sim 10\times$ smaller than RJ. The key observational consequence (Penn 2014 §2.4.1): at fixed spectral resolving power, the IR photon count falls as $\lambda^{-3}$, so S/N drops as $\lambda^{-3/2}$ — the fundamental IR photon-flux penalty.

**한국어.** RJ 근사는 태양 온도에서 $\sim 5\,\mu$m 이상에서만 유효하며, 1.5 μm에서는 완전 플랑크가 RJ보다 10배 작다. 관측적 함의(Penn 2014 §2.4.1): 분해능 일정 시 IR 광자수는 $\lambda^{-3}$로 감소 → S/N은 $\lambda^{-3/2}$ 저하, 이는 IR의 근본적 광자 flux 패널티다.

## 3. Synthetic CO 4.7 μm Line Profile / 합성 CO 4.7 μm 선 프로파일

**English.** The fundamental CO absorption band near 4666 nm (4.7 μm) is the signature of a cool "COmosphere" at heights where Ca II/Mg II H&K emission demands hot plasma (Ayres 1981, 2002). We model a simple Voigt-like absorption line on a Planck continuum.

**한국어.** 4666 nm(4.7 μm) 부근 CO 기본 흡수 밴드는 Ca II/Mg II H&K 방출이 뜨거운 플라스마를 요구하는 높이에서의 차가운 "COmosphere"의 서명이다(Ayres 1981, 2002). 플랑크 연속체 위의 단순 Voigt 유사 흡수선을 모델링한다.

In [ ]:
def pseudo_voigt(x_grid, sigma_gauss, gamma_lorentz):
    """Approximate normalized Voigt profile via pseudo-Voigt linear mix.

    Args:
        x_grid: Array of offsets from line center (same units as sigma, gamma).
        sigma_gauss: Gaussian standard deviation (Doppler broadening).
        gamma_lorentz: Lorentzian HWHM (natural + pressure).

    Returns:
        Pseudo-Voigt profile on the grid (not strictly normalized).
    """
    fg = 2.0 * np.sqrt(2.0 * np.log(2.0)) * sigma_gauss
    fl = 2.0 * gamma_lorentz
    f = (fg**5 + 2.69269 * fg**4 * fl + 2.42843 * fg**3 * fl**2 +
         4.47163 * fg**2 * fl**3 + 0.07842 * fg * fl**4 + fl**5) ** 0.2
    eta = 1.36603 * (fl / f) - 0.47719 * (fl / f) ** 2 + 0.11116 * (fl / f) ** 3
    gauss = np.exp(-0.5 * (x_grid / sigma_gauss) ** 2) / (sigma_gauss * np.sqrt(2.0 * np.pi))
    lorentz = (gamma_lorentz / np.pi) / (x_grid ** 2 + gamma_lorentz ** 2)
    return eta * lorentz + (1.0 - eta) * gauss


wavelength_um = np.linspace(4.660, 4.680, 4000)
line_centers_um = np.array([4.663, 4.668, 4.672, 4.676])
line_depths = np.array([0.55, 0.75, 0.65, 0.48])
sigma_um = 0.0008
gamma_um = 0.0004

spectrum_cool = np.ones_like(wavelength_um)
for center, depth in zip(line_centers_um, line_depths):
    profile = pseudo_voigt(wavelength_um - center, sigma_um, gamma_um)
    profile /= profile.max()
    spectrum_cool -= depth * profile

spectrum_hot = np.ones_like(wavelength_um)
for center, depth in zip(line_centers_um, line_depths):
    profile = pseudo_voigt(wavelength_um - center, sigma_um * 1.2, gamma_um * 1.2)
    profile /= profile.max()
    spectrum_hot -= 0.15 * depth * profile

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(wavelength_um * 1000, spectrum_cool, lw=1.5, color='navy',
        label='Cool COmosphere (T ~ 3500 K): strong CO absorption')
ax.plot(wavelength_um * 1000, spectrum_hot, lw=1.5, color='firebrick',
        label='Hot canopy (T > 5000 K): CO dissociated')
ax.set_xlabel('Wavelength [nm]')
ax.set_ylabel('Normalized intensity I / I_cont')
ax.set_title('Synthetic CO fundamental band near 4666 nm')
ax.legend(fontsize=9)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

**English.** The temperature-sensitivity of CO (it dissociates above $\sim 4000$ K) means *alternating* cool and hot regions produce *alternating* line depths in spectroheliograms. Ayres & Rabin (1996) observations of CO limb extensions (Penn 2014 Fig. 7) reveal exactly this behavior. This diagnostic cannot be reproduced at visible wavelengths because CO has no accessible electronic transitions there.

**한국어.** CO는 $\sim 4000$ K 이상에서 해리되므로, *교차하는* 차가운/뜨거운 영역은 분광영상에서 *교차하는* 선 깊이를 만든다. Ayres & Rabin(1996)의 CO 림 extension 관측(Penn 2014 Fig. 7)이 이 거동을 정확히 보여 준다. 가시광에서는 CO 전자 전이가 접근 불가능하므로 이 진단은 재현될 수 없다.

## 4. Atmospheric Transmission Windows / 대기 투과 창

**English.** Penn Figure 1 reproduces a Kitt Peak transmission atlas showing the J/H/K/L/M bands (1300/1600/2200/3600/5000 nm) separated by strong H$_2$O and CO$_2$ absorption. We model this schematically with Gaussian absorption bands centered at known telluric features.

**한국어.** Penn의 Figure 1은 Kitt Peak에서의 대기 투과율을 보여 준다 — J/H/K/L/M 밴드(1300/1600/2200/3600/5000 nm)가 강한 H$_2$O·CO$_2$ 흡수로 분리되어 있다. 이를 Gaussian 흡수 밴드로 개략적으로 모델링한다.

In [ ]:
def atmospheric_transmission_model(wavelength_nm):
    """Approximate near/mid-IR atmospheric transmission.

    Schematic reproduction of the major H2O/CO2 telluric bands in the
    500-20000 nm range (Penn 2014 Fig. 1).

    Args:
        wavelength_nm: Wavelength grid in nanometers.

    Returns:
        Transmission [0, 1] on the input grid.
    """
    # (center_nm, width_nm, depth) for major absorption bands
    bands = [
        (940,   50,  0.6),
        (1130,  60,  0.65),
        (1400,  90,  0.92),
        (1850, 110,  0.88),
        (2500, 180,  0.95),
        (2700, 100,  0.99),
        (3100, 130,  0.7),
        (4300, 160,  0.97),
        (5500, 250,  0.8),
        (6800, 400,  0.97),
        (9500, 700,  0.85),
        (14000, 1200, 0.97),
        (18000, 1500, 0.97),
    ]
    transmission = np.ones_like(wavelength_nm, dtype=float)
    for center, width, depth in bands:
        transmission *= (1.0 - depth * np.exp(-0.5 * ((wavelength_nm - center) / width) ** 2))
    return np.clip(transmission, 0.0, 1.0)


wavelength_nm_grid = np.linspace(500, 20000, 6000)
trans = atmospheric_transmission_model(wavelength_nm_grid)

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.fill_between(wavelength_nm_grid, 0, trans, color='skyblue', alpha=0.5)
ax.plot(wavelength_nm_grid, trans, color='navy', lw=1)

key_lines = {
    '1083 (He I)': 1083,
    '1075 [Fe XIII]': 1075,
    '1565 (Fe I)': 1565,
    '2231 (Ti I)': 2231,
    '4135 (Fe I)': 4135,
    '4666 (CO)': 4666,
    '12318 (Mg I)': 12318,
}
for lbl, w in key_lines.items():
    ax.axvline(w, color='firebrick', lw=0.8, alpha=0.6)
    ax.annotate(lbl, (w, 1.02), ha='center', va='bottom', fontsize=8,
                rotation=60, color='firebrick')

johnson_bands = {'J': 1300, 'H': 1600, 'K': 2200, 'L': 3600, 'M': 5000}
for lbl, w in johnson_bands.items():
    ax.annotate(lbl, (w, 0.08), fontsize=13, ha='center', color='darkgreen', weight='bold')

ax.set_xscale('log')
ax.set_xlabel('Wavelength [nm]')
ax.set_ylabel('Atmospheric transmission')
ax.set_title('Schematic atmospheric transmission, 0.5-20 um (after Penn 2014 Fig. 1)')
ax.set_ylim(0, 1.15)
ax.set_xlim(500, 20000)
plt.tight_layout()
plt.show()

print('Transmission at Penn key lines:')
for lbl, w in key_lines.items():
    t = atmospheric_transmission_model(np.array([w]))[0]
    print(f'  {lbl:20s} at {w:5d} nm -> T = {t:.2f}')

**English.** The exercise makes vivid that the IR "windows" J/H/K/L/M are not continuous: wavelengths near 1400 nm (strong H$_2$O) and 2500 nm are almost entirely opaque. The Mg I 12 318 nm line sits in a relatively clear region but a space-based observation would be much cleaner. For mid-IR wavelengths above 5 μm the McM-P telescope at Kitt Peak is uniquely suited due to its dry high-altitude site.

**한국어.** 이 연습은 IR "창"이 연속적이지 않음을 생생히 보여 준다 — 1400 nm(강한 H$_2$O) 부근과 2500 nm는 거의 완전 불투명이다. Mg I 12 318 nm 선은 비교적 맑은 구간에 있지만 우주 관측이 훨씬 깨끗할 것이다. 5 μm 이상 중 IR에서는 건조·고도의 Kitt Peak McM-P 망원경이 독보적인 관측지이다.

## 5. Coronal [Fe XIII] 1074.7 nm Visibility Advantage / 코로나 [Fe XIII] 1074.7 nm 가시성 이점

**English.** Off-limb coronal observations are *scattered-light limited*. Because both atmospheric and telescopic scatter scale roughly as $\lambda^{-n}$ with $n = 1.5$-$4$, moving to the IR produces dramatic contrast gains. We simulate the coronal brightness profile above the limb for visible Fe XIV 5303 Å and IR [Fe XIII] 1074.7 nm, showing the relative signal-to-background improvement.

**한국어.** 림 바깥 코로나 관측은 *산란광 한계*에 있다. 대기·망원경 산란 모두 대략 $\lambda^{-n}$($n=1.5$-$4$)로 감소하므로 IR로 이동하면 대비가 극적으로 향상된다. 가시광 Fe XIV 5303 Å과 IR [Fe XIII] 1074.7 nm에 대해 림 바깥 코로나 밝기 프로파일을 시뮬레이션하고 신호-배경 비율 향상을 보인다.

In [ ]:
def coronal_emission(r_Rsun):
    """Schematic coronal emission line brightness vs. heliocentric radius.

    Args:
        r_Rsun: Heliocentric radius in solar radii (>= 1).

    Returns:
        Normalized coronal emission brightness (arbitrary units).
    """
    return 1e-6 * (r_Rsun ** -2.5 + 0.1 * r_Rsun ** -4.0)


def scattered_light(r_Rsun, lambda_nm):
    """Schematic instrumental + atmospheric scatter near the limb.

    Scaling ~ lambda^-2 represents combined Mie + roughness scattering
    (Penn 2014 Section 2.1.2-2.1.3).

    Args:
        r_Rsun: Heliocentric radius (>= 1).
        lambda_nm: Wavelength in nanometers.

    Returns:
        Normalized scattered-light brightness (arbitrary units).
    """
    r_profile = (r_Rsun - 0.98) ** -3 * np.exp(-(r_Rsun - 1.0) * 10)
    wave_factor = (550.0 / lambda_nm) ** 2.0
    return 1e-5 * r_profile * wave_factor


r_values = np.linspace(1.01, 1.5, 250)

sig_vis = coronal_emission(r_values)
bg_vis = scattered_light(r_values, 530.3)
sig_ir = coronal_emission(r_values)
bg_ir = scattered_light(r_values, 1074.7)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

ax = axes[0]
ax.plot(r_values, sig_vis, lw=2, color='green', label='Fe XIV 5303 A coronal emission')
ax.plot(r_values, bg_vis, lw=2, ls='--', color='green', alpha=0.5,
        label='Scattered light (530 nm)')
ax.plot(r_values, sig_ir, lw=2, color='firebrick', label='[Fe XIII] 1074.7 nm emission')
ax.plot(r_values, bg_ir, lw=2, ls='--', color='firebrick', alpha=0.5,
        label='Scattered light (1075 nm)')
ax.set_yscale('log')
ax.set_xlabel('Heliocentric radius [R_sun]')
ax.set_ylabel('Brightness [arbitrary units]')
ax.set_title('Signal vs. scattered-light background')
ax.legend(fontsize=8, loc='upper right')

ax = axes[1]
snr_vis = sig_vis / bg_vis
snr_ir = sig_ir / bg_ir
ax.plot(r_values, snr_vis, lw=2, color='green', label='Visible 5303 A')
ax.plot(r_values, snr_ir, lw=2, color='firebrick', label='IR 1074.7 nm')
ax.set_xlabel('Heliocentric radius [R_sun]')
ax.set_ylabel('Signal / scatter ratio')
ax.set_title('Coronal contrast: IR advantage')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f'Scatter reduction factor (5303 A -> 1074.7 nm): {(1074.7/530.3)**2.0:.2f}x')

**English.** The IR line enjoys about a $(1075/530)^2 \approx 4$-fold scatter reduction over the visible 5303 Å, yielding roughly 4× better contrast at all radii. This is why Tomczyk et al.'s COMP instrument targets [Fe XIII] 1074.7 nm for routine coronal magnetic-field observations and why DKIST's Cryo-NIRSP is the flagship coronal polarimeter.

**한국어.** IR 선은 가시광 5303 Å 대비 약 $(1075/530)^2 \approx 4$배의 산란 감소를 누려 모든 반경에서 약 4배의 대비 이점을 가진다. 이것이 Tomczyk et al.의 COMP 기기가 [Fe XIII] 1074.7 nm를 코로나 자기장의 일상 관측 타깃으로 삼고, DKIST의 Cryo-NIRSP가 대표 코로나 편광기인 이유다.

## 6. Magnetic-Sensitivity Figure of Merit / 자기 민감도 지표

**English.** Penn Table 2 ranks solar spectral lines by $g_{\rm eff}\lambda$ — the magnetic sensitivity figure of merit. We reproduce it and visualize why Mg I 12 318 nm dominates.

**한국어.** Penn Table 2는 태양 스펙트럼 선을 $g_{\rm eff}\lambda$로 순위화한다. 이를 재현하고 Mg I 12 318 nm가 왜 압도적인지 시각화한다.

In [ ]:
# Reproduce Table 2 from Penn (2014) as a list of tuples:
# (ion_name, wavelength_nm, g_eff, atmospheric_region).
table2 = [
    ('Fe I',     525,    3.0,   'Photosphere'),
    ('Fe I',     630,    2.5,   'Photosphere'),
    ('Fe I',     1565,   3.0,   'Photosphere'),
    ('Ti I',     2231,   2.5,   'Photosphere'),
    ('Fe I',     4064,   1.25,  'Photosphere'),
    ('Fe I',     4137,   2.81,  'Photosphere'),
    ('Mg I',     12318,  1.0,   'Photosphere'),
    ('Ca I',     854,    1.1,   'Chromosphere'),
    ('Mg I',     3682,   1.17,  'Chromosphere'),
    ('Ca I',     3697,   1.1,   'Chromosphere'),
    ('[Fe XIV]', 530,    1.33,  'Corona'),
    ('[Fe XIII]',1075,   1.5,   'Corona'),
    ('[Si X]',   3934,   1.5,   'Corona'),
]

labels = [f'{ion} {w}nm' for ion, w, g, r in table2]
wavelengths_nm = np.array([w for ion, w, g, r in table2])
g_effs = np.array([g for ion, w, g, r in table2])
regions = [r for ion, w, g, r in table2]
figure_of_merit = wavelengths_nm * g_effs

region_color = {'Photosphere': 'C0', 'Chromosphere': 'C2', 'Corona': 'C3'}
colors = [region_color[r] for r in regions]

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))

order = np.argsort(figure_of_merit)
ax = axes[0]
ax.barh(np.array(labels)[order], figure_of_merit[order],
        color=np.array(colors)[order])
ax.set_xlabel(r'Magnetic sensitivity  $g_{\rm eff}\lambda$ [nm]')
ax.set_title('Penn Table 2: magnetic sensitivity ranking')
ax.set_xscale('log')

ax = axes[1]
for region, color in region_color.items():
    mask = np.array([r == region for r in regions])
    ax.scatter(wavelengths_nm[mask], g_effs[mask],
               s=figure_of_merit[mask] * 0.1, color=color,
               alpha=0.6, edgecolor='black', linewidth=0.5, label=region)
for lbl, w, g in zip(labels, wavelengths_nm, g_effs):
    ax.annotate(lbl, (w, g), fontsize=7, ha='left', va='bottom', alpha=0.8)
ax.set_xlabel(r'Wavelength $\lambda$ [nm]')
ax.set_ylabel(r'Effective Lande $g_{\rm eff}$')
ax.set_xscale('log')
ax.set_title(r'Lines in $(\lambda, g_{\rm eff})$ space (marker size = $g_{\rm eff}\lambda$)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

best = int(np.argmax(figure_of_merit))
print(f'Most magnetically sensitive line: {labels[best]} '
      f'with g_eff * lambda = {figure_of_merit[best]:.0f} nm')

**English.** Mg I 12 318 nm emerges as the clear winner with $g_{\rm eff}\lambda \approx 12\,300$ nm, about eight times more sensitive than the classic visible Fe I 5250 Å line. This is why Penn (2014 §3.4) calls it "the most sensitive magnetic probe" — though it requires mid-IR instrumentation (12 μm) and can only be observed efficiently at McM-P or comparable dry high-altitude sites.

**한국어.** Mg I 12 318 nm가 $g_{\rm eff}\lambda \approx 12\,300$ nm로 압도적 1위, 고전 가시광 Fe I 5250 Å의 약 8배 민감도를 가진다. 이것이 Penn(2014 §3.4)이 이를 "가장 민감한 자기 탐침"으로 부르는 이유이며, 12 μm 중 IR 기기와 McM-P급 건조·고도 관측지를 요구한다.

## 7. Conclusions / 결론

**English.** Six quantitative demonstrations distilled from Penn (2014):

1. **Zeeman splitting**: Fe I 1565 nm is *fully resolved* at 1 kG (split/Doppler > 4) while optical Fe I 5250 Å stays in the weak-field regime — confirming the $\lambda^2$ advantage.
2. **Planck/RJ**: IR photon count drops as $\lambda^{-3}$ at fixed resolving power; RJ is only valid above $\sim 5\,\mu$m.
3. **CO 4.7 μm**: the fundamental CO band's extreme temperature sensitivity uniquely probes the COmosphere.
4. **Atmospheric windows**: IR transmission is highly structured; Penn's diagnostic lines sit in specific clear windows.
5. **Coronal contrast**: IR [Fe XIII] 1074.7 nm has $\sim 4\times$ better signal/scatter than optical Fe XIV 5303 Å.
6. **Magnetic sensitivity**: Mg I 12 318 nm with $g_{\rm eff}\lambda = 12\,318$ is by far the most sensitive known line.

Together these illustrate why IR has become the preferred regime for accurate solar magnetic-field measurement, and why the DKIST and COSMO facilities were built to exploit it.

**한국어.** Penn(2014)의 여섯 가지 정량적 시연 요약:

1. **제만 분리**: Fe I 1565 nm는 1 kG에서 완전 분리(분리/도플러 > 4), 가시광 Fe I 5250 Å은 약자기장 영역 유지 — $\lambda^2$ 이점 확인.
2. **플랑크/RJ**: 분해능 일정 시 IR 광자수 $\lambda^{-3}$ 감소; RJ는 $\sim 5\,\mu$m 이상에서만 유효.
3. **CO 4.7 μm**: 온도에 극도로 민감한 CO 기본 밴드가 COmosphere를 독특하게 진단.
4. **대기 창**: IR 투과율은 매우 구조화되어 있고, Penn의 진단선들은 특정 맑은 창에 위치.
5. **코로나 대비**: IR [Fe XIII] 1074.7 nm가 가시광 Fe XIV 5303 Å보다 약 4배의 신호/산란 비율.
6. **자기 민감도**: Mg I 12 318 nm($g_{\rm eff}\lambda = 12\,318$)가 알려진 가장 민감한 선.

이 여섯 가지는 IR이 정확한 태양 자기장 측정의 표준 영역이 된 이유, 그리고 DKIST·COSMO가 이를 활용하기 위해 건설된 이유를 함께 설명한다.